In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# ============================================================================
# 1. LOAD DATA
# ============================================================================
print("=" * 80)
print("LOADING DATA...")
print("=" * 80)

# Load data files
orders_df = pd.read_csv('/content/ORDERS (1).csv')
order_items_df = pd.read_csv('/content/ORDER_ITEM (2).csv')
products_df = pd.read_csv('/content/product (1).csv')

print(f"✓ Orders loaded: {orders_df.shape[0]:,} records")
print(f"✓ Order Items loaded: {order_items_df.shape[0]:,} records")
print(f"✓ Products loaded: {products_df.shape[0]:,} records")

# ============================================================================
# 2. DATA PREPROCESSING & MERGING
# ============================================================================
print("\n" + "=" * 80)
print("DATA PREPROCESSING...")
print("=" * 80)

# Convert order_date to datetime
orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])

# Extract Year-Month for grouping
orders_df['year_month'] = orders_df['order_date'].dt.to_period('M')

# Merge order items with orders to get order_date
order_details = order_items_df.merge(
    orders_df[['order_id', 'order_date', 'year_month', 'order_status']],
    on='order_id',
    how='inner'
)

# Merge with products to get category information
order_details = order_details.merge(
    products_df[['product_id', 'product_name', 'category', 'segment', 'price']],
    on='product_id',
    how='inner'
)

# Calculate revenue (quantity * unit_price - discount_amount)
order_details['revenue'] = (order_details['quantity'] * order_details['unit_price']) - order_details['discount_amount']

# Filter only delivered orders (successful transactions)
order_details_delivered = order_details[order_details['order_status'] == 'delivered'].copy()

print(f"✓ Total order items: {order_details.shape[0]:,}")
print(f"✓ Delivered orders: {order_details_delivered.shape[0]:,}")
print(f"✓ Date range: {orders_df['order_date'].min().date()} to {orders_df['order_date'].max().date()}")
print(f"✓ Number of product categories: {products_df['category'].nunique()}")

# ============================================================================
# 3. AGGREGATE REVENUE BY CATEGORY AND MONTH
# ============================================================================
print("\n" + "=" * 80)
print("CALCULATING MONTHLY REVENUE BY CATEGORY...")
print("=" * 80)

# Group by category and month
monthly_category_revenue = order_details_delivered.groupby(['category', 'year_month'])['revenue'].sum().reset_index()
monthly_category_revenue.columns = ['category', 'year_month', 'monthly_revenue']
monthly_category_revenue = monthly_category_revenue.sort_values(['category', 'year_month'])

# Pivot table for analysis
revenue_pivot = monthly_category_revenue.pivot(
    index='year_month',
    columns='category',
    values='monthly_revenue'
).fillna(0)

print(f"✓ Monthly revenue data created: {revenue_pivot.shape[0]} months × {revenue_pivot.shape[1]} categories")
print(f"\nCategories: {list(revenue_pivot.columns)}")

# ============================================================================
# 4. CALCULATE STABILITY METRICS
# ============================================================================
print("\n" + "=" * 80)
print("CALCULATING STABILITY METRICS...")
print("=" * 80)

# Initialize results dataframe
stability_metrics = pd.DataFrame(index=revenue_pivot.columns)
stability_metrics.index.name = 'category'

# Calculate Measure: Total Revenue
stability_metrics['total_revenue'] = revenue_pivot.sum()

# Calculate Measure: Average Monthly Revenue
stability_metrics['avg_monthly_revenue'] = revenue_pivot.mean()

# Calculate Measure: Min and Max Monthly Revenue
stability_metrics['min_monthly_revenue'] = revenue_pivot.min()
stability_metrics['max_monthly_revenue'] = revenue_pivot.max()

# Calculate Metric: Variance
stability_metrics['variance'] = revenue_pivot.var()

# Calculate Metric: Standard Deviation
stability_metrics['std_deviation'] = revenue_pivot.std()

# Calculate Metric: Coefficient of Variation (CV) = StdDev / Mean
# CV < 0.20: Stable, 0.20-0.40: Moderate, 0.40-0.60: High, > 0.60: Very High
stability_metrics['coefficient_of_variation'] = (
    stability_metrics['std_deviation'] / stability_metrics['avg_monthly_revenue']
).round(4)

# Calculate Metric: Revenue Trend (linear regression slope)
# Positive: growth, Negative: decline
def calculate_trend(series):
    """Calculate trend using linear regression"""
    x = np.arange(len(series))
    y = series.values
    if len(x) > 1:
        z = np.polyfit(x, y, 1)
        return z[0]  # Return slope
    return 0

stability_metrics['trend_slope'] = revenue_pivot.apply(calculate_trend, axis=0)

# Calculate Metric: Month-to-Month Growth Rate (average)
def calculate_growth_rate(series):
    """Calculate average month-to-month growth rate"""
    pct_change = series.pct_change().dropna()
    if len(pct_change) > 0:
        return pct_change.mean()
    return 0

stability_metrics['avg_growth_rate'] = revenue_pivot.apply(calculate_growth_rate, axis=0).round(4)

# Calculate KPI: Range (Max - Min)
stability_metrics['revenue_range'] = stability_metrics['max_monthly_revenue'] - stability_metrics['min_monthly_revenue']

# Calculate KPI: Ratio Range / Average (Volatility Index)
stability_metrics['volatility_index'] = (
    stability_metrics['revenue_range'] / stability_metrics['avg_monthly_revenue']
).round(4)

# Sort by coefficient of variation
stability_metrics = stability_metrics.sort_values('coefficient_of_variation', ascending=False)

# ============================================================================
# 5. CLASSIFY CATEGORIES
# ============================================================================
print("\n" + "=" * 80)
print("CLASSIFYING PRODUCT CATEGORIES...")
print("=" * 80)

def classify_category(row):
    """
    Classify category based on multiple metrics:
    1. Coefficient of Variation (stability)
    2. Trend Slope (growth/decline)
    3. Growth Rate
    """
    cv = row['coefficient_of_variation']
    trend = row['trend_slope']
    growth = row['avg_growth_rate']

    # Determine stability based on CV
    if cv < 0.20:
        stability = "Stable"
    elif cv < 0.40:
        stability = "Moderate"
    elif cv < 0.60:
        stability = "High Volatility"
    else:
        stability = "Very High Volatility"

    # Determine trend
    if trend > 0 and growth > 0.01:
        trend_type = "Growth"
        classification = f"{stability} - Growth Trend"
    elif trend < -0.1 or growth < -0.01:
        trend_type = "Decline"
        classification = f"{stability} - Declining Trend"
    else:
        trend_type = "Stable"
        classification = f"{stability} - Stable Trend"

    return classification

stability_metrics['classification'] = stability_metrics.apply(classify_category, axis=1)

# ============================================================================
# 6. IDENTIFY STRATEGIC PRIORITIES
# ============================================================================
print("\n" + "=" * 80)
print("IDENTIFYING STRATEGIC PRIORITIES...")
print("=" * 80)

def assign_priority(row):
    """
    Assign priority for investment/improvement based on:
    1. Total revenue contribution
    2. Volatility
    3. Trend
    4. Growth rate
    """
    classification = row['classification']
    cv = row['coefficient_of_variation']
    revenue = row['total_revenue']
    growth = row['avg_growth_rate']

    # Normalize metrics for scoring (0-100)
    revenue_score = (revenue / stability_metrics['total_revenue'].max()) * 100
    stability_score = (1 - min(cv / 1.0, 1)) * 100  # Higher if more stable
    growth_score = max(growth * 500, 0)  # Scale growth rate

    # Calculate priority score
    priority_score = (revenue_score * 0.4) + (stability_score * 0.3) + (growth_score * 0.3)

    # Determine priority tier
    if "Growth" in classification and cv < 0.40:
        priority = "HIGH - Growing & Stable"
        score_rank = 1
    elif "Growth" in classification and cv < 0.60:
        priority = "MEDIUM-HIGH - Growing but Volatile"
        score_rank = 2
    elif "Stable" in classification and revenue > stability_metrics['total_revenue'].median():
        priority = "MEDIUM - Stable Revenue Generator"
        score_rank = 3
    elif "Declining" in classification and cv < 0.40:
        priority = "MEDIUM - Stable but Declining (Investigate)"
        score_rank = 4
    elif "High" in classification or "Very High" in classification:
        priority = "MEDIUM-LOW - Needs Strategy Review"
        score_rank = 5
    else:
        priority = "LOW - Monitor"
        score_rank = 6

    return pd.Series({'priority': priority, 'priority_score': priority_score, 'score_rank': score_rank})

priority_df = stability_metrics.apply(assign_priority, axis=1)
stability_metrics = pd.concat([stability_metrics, priority_df], axis=1)

# ============================================================================
# 7. DISPLAY RESULTS
# ============================================================================
print("\n" + "=" * 80)
print("REVENUE STABILITY ANALYSIS RESULTS")
print("=" * 80)

# Display full metrics table
print("\n DETAILED STABILITY METRICS:")
print("-" * 150)

display_columns = [
    'total_revenue', 'avg_monthly_revenue', 'min_monthly_revenue', 'max_monthly_revenue',
    'coefficient_of_variation', 'volatility_index', 'trend_slope', 'avg_growth_rate', 'classification', 'priority'
]

results_display = stability_metrics[display_columns].copy()
results_display['total_revenue'] = results_display['total_revenue'].apply(lambda x: f"{x:,.0f}")
results_display['avg_monthly_revenue'] = results_display['avg_monthly_revenue'].apply(lambda x: f"{x:,.0f}")
results_display['min_monthly_revenue'] = results_display['min_monthly_revenue'].apply(lambda x: f"{x:,.0f}")
results_display['max_monthly_revenue'] = results_display['max_monthly_revenue'].apply(lambda x: f"{x:,.0f}")

print(results_display.to_string())

# ============================================================================
# 8. STRATEGIC RECOMMENDATIONS
# ============================================================================
print("\n" + "=" * 80)
print("STRATEGIC RECOMMENDATIONS")
print("=" * 80)

# Group by priority
for priority_level in ["HIGH - Growing & Stable", "MEDIUM-HIGH - Growing but Volatile",
                       "MEDIUM - Stable Revenue Generator", "MEDIUM - Stable but Declining (Investigate)",
                       "MEDIUM-LOW - Needs Strategy Review", "LOW - Monitor"]:
    categories = stability_metrics[stability_metrics['priority'] == priority_level].index.tolist()
    if categories:
        print(f"\n {priority_level}:")
        for cat in categories:
            cv = stability_metrics.loc[cat, 'coefficient_of_variation']
            growth = stability_metrics.loc[cat, 'avg_growth_rate']
            revenue = stability_metrics.loc[cat, 'total_revenue']
            print(f"   • {cat}: ${revenue:,.0f} total | CV: {cv:.4f} | Avg Growth: {growth:.2%}")

# ============================================================================
# 9. EXPORT RESULTS TO CSV
# ============================================================================
print("\n" + "=" * 80)
print("EXPORTING RESULTS...")
print("=" * 80)

# Export detailed metrics
stability_metrics_export = stability_metrics.copy()
stability_metrics_export.to_csv('revenue_stability_metrics.csv', index=True)
print("✓ Exported: revenue_stability_metrics.csv")

# Export monthly revenue by category
revenue_pivot.to_csv('monthly_revenue_by_category.csv')
print("✓ Exported: monthly_revenue_by_category.csv")

# Export classification summary
classification_summary = stability_metrics[['classification', 'priority', 'coefficient_of_variation',
                                           'avg_growth_rate', 'total_revenue']].copy()
classification_summary.to_csv('category_classification.csv')
print("✓ Exported: category_classification.csv")

# ============================================================================
# 10. VISUALIZATION
# ============================================================================
print("\n" + "=" * 80)
print("CREATING VISUALIZATIONS...")
print("=" * 80)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Monthly Revenue Trend by Category
ax1 = axes[0, 0]
for column in revenue_pivot.columns:
    ax1.plot(range(len(revenue_pivot)), revenue_pivot[column].values, marker='o', label=column, linewidth=2)
ax1.set_xlabel('Month', fontsize=11, fontweight='bold')
ax1.set_ylabel('Revenue', fontsize=11, fontweight='bold')
ax1.set_title('Monthly Revenue Trend by Category', fontsize=12, fontweight='bold')
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.grid(True, alpha=0.3)

# 2. Coefficient of Variation Comparison
ax2 = axes[0, 1]
cv_sorted = stability_metrics['coefficient_of_variation'].sort_values(ascending=True)
colors = ['green' if x < 0.20 else 'orange' if x < 0.40 else 'red' for x in cv_sorted.values]
ax2.barh(range(len(cv_sorted)), cv_sorted.values, color=colors)
ax2.set_yticks(range(len(cv_sorted)))
ax2.set_yticklabels(cv_sorted.index)
ax2.set_xlabel('Coefficient of Variation', fontsize=11, fontweight='bold')
ax2.set_title('Revenue Stability by Category\n(Lower CV = More Stable)', fontsize=12, fontweight='bold')
ax2.axvline(x=0.20, color='green', linestyle='--', alpha=0.5, label='Stable Threshold')
ax2.axvline(x=0.40, color='orange', linestyle='--', alpha=0.5, label='Moderate Threshold')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='x')

# 3. Total Revenue by Category
ax3 = axes[1, 0]
revenue_sorted = stability_metrics['total_revenue'].sort_values(ascending=True)
ax3.barh(range(len(revenue_sorted)), revenue_sorted.values, color='skyblue')
ax3.set_yticks(range(len(revenue_sorted)))
ax3.set_yticklabels(revenue_sorted.index)
ax3.set_xlabel('Total Revenue', fontsize=11, fontweight='bold')
ax3.set_title('Total Revenue Contribution by Category', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='x')

# 4. Growth Rate vs Coefficient of Variation (Scatter)
ax4 = axes[1, 1]
scatter = ax4.scatter(stability_metrics['coefficient_of_variation'],
                      stability_metrics['avg_growth_rate'] * 100,
                      s=stability_metrics['total_revenue'] / 1000,
                      alpha=0.6, c=range(len(stability_metrics)), cmap='viridis')

for idx, cat in enumerate(stability_metrics.index):
    ax4.annotate(cat,
                (stability_metrics.loc[cat, 'coefficient_of_variation'],
                 stability_metrics.loc[cat, 'avg_growth_rate'] * 100),
                fontsize=9, alpha=0.8)

ax4.set_xlabel('Coefficient of Variation (Volatility)', fontsize=11, fontweight='bold')
ax4.set_ylabel('Average Growth Rate (%)', fontsize=11, fontweight='bold')
ax4.set_title('Category Positioning: Stability vs Growth\n(Bubble size = Total Revenue)', fontsize=12, fontweight='bold')
ax4.axhline(y=0, color='red', linestyle='--', alpha=0.5)
ax4.axvline(x=0.20, color='green', linestyle='--', alpha=0.5)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('revenue_stability_analysis.png', dpi=300, bbox_inches='tight')
print("✓ Saved: revenue_stability_analysis.png")
plt.close()

# ============================================================================
# 11. SUMMARY STATISTICS
# ============================================================================
print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)

print(f"\n OVERALL METRICS:")
print(f"   • Total Revenue across all categories: ${stability_metrics['total_revenue'].sum():,.0f}")
print(f"   • Average monthly revenue (all categories): ${stability_metrics['avg_monthly_revenue'].mean():,.0f}")
print(f"   • Most stable category (lowest CV): {stability_metrics['coefficient_of_variation'].idxmin()} (CV: {stability_metrics['coefficient_of_variation'].min():.4f})")
print(f"   • Most volatile category (highest CV): {stability_metrics['coefficient_of_variation'].idxmax()} (CV: {stability_metrics['coefficient_of_variation'].max():.4f})")

# Count classifications
print(f"\n CLASSIFICATION BREAKDOWN:")
for classification in stability_metrics['classification'].unique():
    count = (stability_metrics['classification'] == classification).sum()
    revenue = stability_metrics[stability_metrics['classification'] == classification]['total_revenue'].sum()
    print(f"   • {classification}: {count} category/categories (${revenue:,.0f})")

# Count priorities
print(f"\n PRIORITY DISTRIBUTION:")
for priority in ["HIGH - Growing & Stable", "MEDIUM-HIGH - Growing but Volatile",
                 "MEDIUM - Stable Revenue Generator", "MEDIUM - Stable but Declining (Investigate)",
                 "MEDIUM-LOW - Needs Strategy Review", "LOW - Monitor"]:
    count = (stability_metrics['priority'] == priority).sum()
    if count > 0:
        print(f"   • {priority}: {count} category/categories")

print("\n" + "=" * 80)
print("✓ ANALYSIS COMPLETE!")
print("=" * 80)
print("\nGenerated files:")
print("   1. revenue_stability_metrics.csv - Detailed metrics for all categories")
print("   2. monthly_revenue_by_category.csv - Month-by-month revenue data")
print("   3. category_classification.csv - Classification and priority summary")
print("   4. revenue_stability_analysis.png - Visualization charts")
print("\n" + "=" * 80)


LOADING DATA...
✓ Orders loaded: 646,945 records
✓ Order Items loaded: 714,669 records
✓ Products loaded: 2,412 records

DATA PREPROCESSING...
✓ Total order items: 714,669
✓ Delivered orders: 570,887
✓ Date range: 2012-07-04 to 2022-12-31
✓ Number of product categories: 4

CALCULATING MONTHLY REVENUE BY CATEGORY...
✓ Monthly revenue data created: 126 months × 4 categories

Categories: ['Casual', 'GenZ', 'Outdoor', 'Streetwear']

CALCULATING STABILITY METRICS...

CLASSIFYING PRODUCT CATEGORIES...

IDENTIFYING STRATEGIC PRIORITIES...

REVENUE STABILITY ANALYSIS RESULTS

 DETAILED STABILITY METRICS:
------------------------------------------------------------------------------------------------------------------------------------------------------
             total_revenue avg_monthly_revenue min_monthly_revenue max_monthly_revenue  coefficient_of_variation  volatility_index    trend_slope  avg_growth_rate                          classification                            priority
catego